# Random Forest Regression - default hyperparameters
for Body mass prediction in biped dinosaurs using supervised learning
-----------------------------------------------------------------------------------------------------------------------------
Alejandro Otero - CONICET - Museo de La Plata (UNLP), Argentina. alexandros.otero@gmail.com

Ailen Borya - Especialización en Geociencias de Exploración y Desarrollo de Hidrocarburos (UNLP), Argentina. ailenborya@gmail.com

## 1. Libraries

In [ ]:
# Libraries

#!pip install ipywidgets
#!pip install matplotlib
#!pip install seaborn
#!pip install plotly
#!pip install statsmodels
#%pip install pydotplus
#!pip install pandas
!pip install pillow
!pip install shap
!pip install eli5
#!pip install -U kaleido
!pip install openpyxl
!pip install imbalanced-learn
!pip install smogn
#%pip install tensorflow

import os
import math
import pandas as pd
import shap
import eli5
import smogn
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits import mplot3d
import numpy as np
import seaborn as sns
#from plotly.offline import plot
#import plotly.express as px
#import plotly.graph_objects as go
#import plotly.io as pio
#import statsmodels.api as sm
#from sklearn.inspection import permutation_importance
#from eli5.sklearn import PermutationImportance
from sklearn.model_selection import validation_curve
#from sklearn import preprocessing
#from sklearn.preprocessing import normalize
#from sklearn.preprocessing import MinMaxScaler
#from sklearn.preprocessing import StandardScaler
#from sklearn.preprocessing import RobustScaler
#from sklearn.model_selection import GridSearchCV
%matplotlib inline

from sklearn import linear_model
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
#from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
#from sklearn.cluster import KMeans
#import tkinter as tk

from sklearn.model_selection import train_test_split
from sklearn import metrics

#from sklearn import svm
from six import StringIO
from sklearn.tree import export_graphviz

#import pydotplus
#from pydotplus import graph_from_dot_data
#from sklearn.datasets import make_blobs
from imblearn.over_sampling import SMOTE

## 2. Data preprocessing (basic)

In [ ]:
# Maximum columns to display

pd.options.display.max_columns = 100

In [ ]:
# Suppress scientific notation by setting float_format

pd.options.display.float_format = '{:.2f}'.format

In [ ]:
# Directory location

os.getcwd()

In [ ]:
# Colab open file
'''
from google.colab import files
uploaded = files.upload()
'''

In [ ]:
# Open file

df = pd.read_excel('train_data.xlsx')
df.head(5) # Visualising first 5 rows

In [ ]:
# Convert all numerical variables to natural logarithm 

for col in df.select_dtypes(include = np.number).columns:
    df[col] = np.log(df[col])

# Visualising dataset

df.head()

## 3. Data split

In [ ]:
# Store predictor variables in X
# Store target variable in y

X = df.filter([
    'Femur length (mm)',
    'Femur circumference (mm)'
])
y = df['Body mass (g)']

In [ ]:
# Train and Test split

X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size = 0.25, # split ratio 72:25
    random_state = 42 # enables replication
)

## 4. Model creation and fit

In [ ]:
# Create Model instance

%timeit

forest = RandomForestRegressor(
    random_state = 42
)

# Model Fitting

forest.fit(
    X_train,
    y_train
)

## 5. Model prediction and performance

In [ ]:
# Prediction on test data

y_pred_test = forest.predict(X_test) # still in log

# Retrotransformation
# back to BM_g

y_pred = np.exp(y_pred_test) # back to predicted values (in grams)
y_true = np.exp(y_test) # real values (in grams) ---> back to the original y_test

In [ ]:
# Model prediction and performance

# Mean Squared Error (MSE)
# It calculates the average of the squares of the errors, which gives a higher weight to larger errors.

mse_test = (mean_squared_error(y_true, y_pred))

# R-squared (Coefficient of Determination)
# It indicates the proportion of variance in the dependent variable that can be predicted from the independent variables.

#r2_train = r2_score(y_train_s, y_pred_train)
r2_test = r2_score(y_true, y_pred)

#print('Mean squared error MSE_train: ', mse_train)
print('Mean squared error MSE_test: ', mse_test)
print('\n')

#print('R2_train: ', r2_train)
print('R2_test: ', r2_test)

## 6. Predicting new data on validation set

In [ ]:
# Open validation file

df_val = pd.read_excel('val_data.xlsx')
df_val.head()

In [ ]:
# Convert all numberical variables to natural logarithm

for col in df_val.select_dtypes(include = np.number).columns:
    df_val[col] = np.log(df_val[col])

df_val.head(5)

In [ ]:
# Store predictor variables in X
# Store target variable in y

X_val = df_val.filter([
    'Femur length (mm)',
    'Femur circumference (mm)'
])
y_val = df_val['Body mass (g)']

In [ ]:
# Prediction new value

y_val_pred = forest.predict(X_val)

In [ ]:
# Retrotransform - back to BM_g

y_val_pred = np.exp(y_val_pred) # predicted values (in grams)
y_val_true = np.exp(y_val) # real values (in grams) ---> the original y_val

## 7. Assessing performance on validation set

In [ ]:
# Model prediction and performance

# Mean Squared Error (MSE)
# It calculates the average of the squares of the errors, which gives a higher weight to larger errors.

mse_val = (mean_squared_error(y_val_true, y_val_pred))

# R-squared (Coefficient of Determination)
# It indicates the proportion of variance in the dependent variable that can be predicted from the independent variables.

r2_val = r2_score(y_val_true, y_val_pred)

print('Mean squared error MSE_val: ', mse_val)
print('\n')
print('R2_val: ', r2_val)

In [ ]:
## Ratio estimator (Snowdon, 1991)
# mean sample original / mean back-transformed predicted values

RE = y_val_true.mean() / y_val_pred.mean()
y_val_pred_RE = y_val_pred*RE
RE

In [ ]:
# Convert predictions to dataframe

y_val_pred_RE = pd.DataFrame(y_val_pred_RE)
y_val_pred_RE.head()

In [ ]:
# Save predictions in excel file

y_val_pred_RE.to_excel('val_pred.xlsx', float_format = '%.2f')

## 8. Feature importance on validation set

In [ ]:
# SHAP

# Use TreeExplainer for tree-based models
explainer_val = shap.Explainer(forest, X_train) # can use also X_train

# Calculate SHAP values for a subset (e.g. test set)
# The predictions are the output of y_pred_test
shap_values_val = explainer_val(X_val) # can use also X_test

# Summary plot (bar chart of average absolute SHAP values)
#shap.plots.bar(shap_values)

# Compute mean absolute SHAP values
mean_abs_shap_val = np.abs(shap_values_val.values).mean(axis = 0)

# Create a DataFrame for sorting
importance_df_val = pd.DataFrame({
    'feature': shap_values_val.feature_names,
    'importance': mean_abs_shap_val
}).sort_values('importance', ascending = True)

# Custom bar plot with green color
plt.figure(figsize = (6, 2.5))
plt.barh(importance_df_val['feature'], importance_df_val['importance'], color = 'black')
plt.xlabel("Mean |SHAP value|")
plt.title("RFdh SHAP (validation set)")
plt.tight_layout()
#plt.savefig('RFdh_shap_val_extant.jpg', dpi = 500, bbox_inches = 'tight')
plt.show()

In [ ]:
fig_val = shap.plots.beeswarm(
    shap_values_val, 
    show = False,
    color = plt.get_cmap('jet'))

#plt.savefig('RFdh_shap2_val_extant.jpg', dpi = 500, bbox_inches = 'tight')

plt.show()

In [ ]:
shap.plots.initjs()

In [ ]:
# visualize the first prediction's explanation with a force plot
shap.plots.force(shap_values_val[7])
#plt.title("RFdh SHAP force")
#plt.savefig('RFdh_force.jpg', dpi = 500, bbox_inches = 'tight')

## 9. Predicting biped dinosaurs (dataset from Campione et al., 2014, table 3)

Using only femoral circumference (FC) to compre with cQE

In [ ]:
# Open  file

df_bip = pd.read_excel('val_bipeds.xlsx')
df_bip.head()

In [ ]:
# Removing non-numerical columns

df_bip = df_bip.iloc[:,[1]]
df_bip.head(5)

In [ ]:
# Convert all numerical variables to natural logarithm

df_bip_log = np.log(df_bip)
df_bip_log.head()

In [ ]:
# Store predictor variables in X (FL locked. Unlock and lock features based on the preferred predictor combinations to use)
# FC is the only predictor used to be able to compare with cQE
# Store target variable in y

# Store predictor variables in X
# Store target variable in y

X = df.filter([
    #'Femur length (mm)',
    'Femur circumference (mm)'
])
y = df['Body mass (g)']

In [ ]:
# Train and Test split

X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size = 0.25, 
    random_state = 42
)

In [ ]:
# Create Model instance

%timeit

forest = RandomForestRegressor(
    random_state = 42
)

# Model Fitting

forest.fit(
    X_train,
    y_train
)

In [ ]:
# Prediction new value

df_bip_pred = forest.predict(df_bip_log)

In [ ]:
# Retrotransform - back to BM

y_bip_pred = np.exp(df_bip_pred) # predicted values (in grams)

In [ ]:
# Convert predictions to dataframe

df_bip_pred = pd.DataFrame(y_bip_pred)
df_bip_pred.head()

In [ ]:
df_bip_pred.to_excel('pred_bip.xlsx', float_format = '%.2f')

## End code